> ⚠️ **INSTRUCTOR TESTING COPY — not for students.**  
> Student-blank cells (Ψ⁻ circuit, `compute_correlation`, Ψ⁻ X-basis) are filled in below, each wrapped between `>>> INSTRUCTOR SOLUTION` / `<<< END INSTRUCTOR SOLUTION` markers. Search for `INSTRUCTOR SOLUTION` before promoting this to the student version.  
> Source: `final_phase1_testing.ipynb` — do not distribute.

# Final Project — Phase 1: Bell States on Real Hardware
**PHYS 349 — Individual Assignment (50 pts)**

---

**Due: Wednesday, April 22 at midnight**

This is the individual warm-up for the final project. Everyone does the same thing: prepare **two** Bell states on a real IBM quantum processor, measure them in two different bases (ZZ and XX), and compare the results to what a perfect simulator predicts.

You already know Bell states cold from Chapter 3. The goal here isn't new physics — it's to go end-to-end on real hardware, see how much reality deviates from the ideal, and get comfortable with the IBM Qiskit Runtime workflow before your team project starts next week.

---

**How this notebook is organized:**

The first half (Parts 1 and 2) runs entirely on the **local simulator** — no IBM account needed. Build your circuits, debug them, and make sure everything works before you touch real hardware.

The second half sends your circuits to a **real IBM quantum processor**. You'll set up your IBM account when you get there (instructions included). Don't skip ahead — develop on the simulator first.

> ⚠️ **Develop on the simulator first.** Do not submit to real hardware until your circuits work correctly on `AerSimulator`. Queue times on real hardware can be minutes to hours, and every run costs you QPU time.

In [ ]:
# === Standard imports ===
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
import numpy as np
import matplotlib.pyplot as plt

# Local simulator for debugging + comparison (no IBM account needed)
simulator = AerSimulator()

SHOTS = 4096
print("✓ Imports loaded. Simulator ready.")

---
## Part 1: Prepare and Measure Two Bell States in the Z Basis (20 pts)

You'll prepare **two** Bell states and measure each in the Z basis (the standard computational basis):

| State | Formula | Recipe |
|:-----:|:--------|:-------|
| $\lvert\Phi^+\rangle$ | $\frac{1}{\sqrt{2}}(\lvert 00\rangle + \lvert 11\rangle)$ | H on q0, CNOT 0→1 |
| $\lvert\Psi^-\rangle$ | $\frac{1}{\sqrt{2}}(\lvert 01\rangle - \lvert 10\rangle)$ | H on q0, CNOT 0→1, X on q1, Z on q0 |

These two are chosen because they differ by more than one gate — so you'll see the effect of accumulating a couple of extra gates on hardware noise.

**Do the steps in order:**
1. Build both circuits.
2. Sanity-check with `Statevector`: does the state match theory?
3. Run 4096 shots on `AerSimulator`: do the histograms look right?
4. **Only then** transpile for the real backend and submit.
5. Plot simulator vs. hardware histograms side by side.

In [ ]:
# === Φ+ : Worked example (do not modify) ===
# Use this as your template for the Ψ- circuit below.

def build_phi_plus():
    """Prepare |Φ+⟩ = (|00⟩ + |11⟩)/√2 and measure in the Z basis."""
    qc = QuantumCircuit(2, 2)
    qc.h(0)
    qc.cx(0, 1)
    qc.measure([0, 1], [0, 1])
    return qc

phi_plus = build_phi_plus()
phi_plus.draw(output='mpl', style='iqp')

In [ ]:
# === Ψ- : Your turn ===
# Build a circuit that prepares |Ψ-⟩ = (|01⟩ - |10⟩)/√2 and measures in the Z basis.
# Recipe: H on q0, CNOT 0→1, X on q1, Z on q0, then measure.

def build_psi_minus():
    """Prepare |Ψ-⟩ = (|01⟩ - |10⟩)/√2 and measure in the Z basis."""
    qc = QuantumCircuit(2, 2)
    # >>> INSTRUCTOR SOLUTION — remove before distributing >>>
    qc.h(0)
    qc.cx(0, 1)
    qc.x(1)
    qc.z(0)
    # <<< END INSTRUCTOR SOLUTION <<<
    qc.measure([0, 1], [0, 1])
    return qc

psi_minus = build_psi_minus()
psi_minus.draw(output='mpl', style='iqp')

In [ ]:
# === Statevector sanity check ===
# Before spending QPU time, verify both circuits produce the correct state.
# We strip off the measurements and compare to the theoretical statevector.

def strip_measurements(qc):
    """Return a copy of qc with measurements removed."""
    return qc.remove_final_measurements(inplace=False)

# Theoretical target states
phi_plus_ideal  = Statevector([1, 0, 0, 1]) / np.sqrt(2)
psi_minus_ideal = Statevector([0, 1, -1, 0]) / np.sqrt(2)

# Simulate the circuits you built
phi_plus_sv  = Statevector(strip_measurements(phi_plus))
psi_minus_sv = Statevector(strip_measurements(psi_minus))

print("Φ+ matches theory: ", phi_plus_sv.equiv(phi_plus_ideal))
print("Ψ- matches theory: ", psi_minus_sv.equiv(psi_minus_ideal))

assert phi_plus_sv.equiv(phi_plus_ideal), "Φ+ circuit does not produce (|00⟩+|11⟩)/√2"
assert psi_minus_sv.equiv(psi_minus_ideal), "Ψ- circuit does not produce (|01⟩-|10⟩)/√2"
print("\n✓ Both circuits verified. Safe to send to hardware.")

In [ ]:
# === Run on the local simulator (no QPU time used) ===
# If either histogram looks wrong, FIX THE CIRCUIT before continuing.

sim_counts = {
    'Phi+' : simulator.run(phi_plus,  shots=SHOTS).result().get_counts(),
    'Psi-' : simulator.run(psi_minus, shots=SHOTS).result().get_counts(),
}

for name, counts in sim_counts.items():
    print(f"{name}: {counts}")

# Expected:
#   Φ+  →  ~50/50 between '00' and '11'; '01' and '10' ≈ 0
#   Ψ-  →  ~50/50 between '01' and '10'; '00' and '11' ≈ 0
plot_histogram([sim_counts['Phi+'], sim_counts['Psi-']],
               legend=['|Φ+⟩ sim', '|Ψ-⟩ sim'])

---
## Connect to IBM Quantum Hardware

Everything above runs on the local simulator. From here on you'll send circuits to a **real quantum processor** on IBM Cloud.

**One-time setup (do this once per laptop):**

1. Go to [quantum.cloud.ibm.com](https://quantum.cloud.ibm.com) and log in (Google OAuth with your university or personal Gmail is fastest).
2. On first login, a "Welcome" modal appears. Leave the defaults (`open-instance`, `us-east`, Open Plan) and click **Create instance**. This gives you 10 free QPU minutes per month (no credit card needed).
3. Create an **API key**: click **Create +** in the API key section on the home page. **Download the key immediately** — it is shown only once and disappears after ~5 minutes.
4. Copy your **instance CRN**: go to Instances → click `open-instance` → copy the CRN. It is a long string starting with `crn:v1:bluemix:...` and ending with `::` (the double colon is correct, don't trim it).
5. Paste your API key and CRN into the cell below and run it **once**. After that, your credentials are saved locally and you never need to enter them again.

**Common gotchas:**
- If you see "Unable to find a default account," you haven't run the save cell below yet (or ran it in a different Python environment).
- If `.backends()` returns an empty list, you skipped step 2 (creating the instance).
- If you get "Could not verify IBM Cloud credentials," double-check your API key and make sure the CRN ends with `::`.

In [ ]:
# === IBM Account Setup (run once, then re-comment) ===
# Uncomment the lines below, paste your API key and CRN, and run this cell ONCE.
# After that, re-comment these lines — your credentials are saved at ~/.qiskit/qiskit-ibm.json.
#
# To find these values:
#   API key  → quantum.cloud.ibm.com → API key section → Create + → Download
#   CRN      → quantum.cloud.ibm.com → Instances → open-instance → copy CRN

# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_cloud",
#     token="YOUR_API_KEY_HERE",
#     instance="YOUR_INSTANCE_CRN_HERE",
#     overwrite=True,
#     set_as_default=True,
# )

In [ ]:
# === Connect to IBM Quantum ===
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

service = QiskitRuntimeService()    # picks up saved credentials automatically
backend = service.least_busy(operational=True, simulator=False)
print(f"Connected! Using backend: {backend.name}  ({backend.num_qubits} qubits)")

In [ ]:
# === Transpile for the real backend and submit Z-basis circuits ===
# >>> INSTRUCTOR SIM SUBSTITUTE — revert before distributing >>>
# Student version submits to Sampler(mode=backend). For instructor testing we
# use AerSimulator with a noise model pulled from the real backend so we get
# hardware-like results synchronously, without QPU time or queue wait.
from qiskit_aer.noise import NoiseModel

pm = generate_preset_pass_manager(target=backend.target, optimization_level=3)
phi_plus_t  = pm.run(phi_plus)
psi_minus_t = pm.run(psi_minus)
print(f"Φ+ transpiled depth: {phi_plus_t.depth()}")
print(f"Ψ- transpiled depth: {psi_minus_t.depth()}")

noise_model = NoiseModel.from_backend(backend)
noisy_sim   = AerSimulator(noise_model=noise_model)

hw_counts = {
    'Phi+': noisy_sim.run(phi_plus_t,  shots=SHOTS).result().get_counts(),
    'Psi-': noisy_sim.run(psi_minus_t, shots=SHOTS).result().get_counts(),
}
print(f"\n(noisy-sim substitute) Φ+ counts: {hw_counts['Phi+']}")
print(f"(noisy-sim substitute) Ψ- counts: {hw_counts['Psi-']}")
# <<< END INSTRUCTOR SIM SUBSTITUTE <<<

In [ ]:
# === Retrieve hardware results ===
# >>> INSTRUCTOR SIM SUBSTITUTE — in the student version this retrieves job_part1 >>>
# hw_counts was already populated above by the noisy simulator; this cell just
# re-prints for parity with the student flow.
for name, counts in hw_counts.items():
    print(f"{name}: {counts}")
# <<< END INSTRUCTOR SIM SUBSTITUTE <<<

In [ ]:
# === Plot simulator vs hardware, side by side ===
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for i, name in enumerate(['Phi+', 'Psi-']):
    plot_histogram(sim_counts[name], ax=axes[i, 0], title=f"|{name}⟩ — Simulator")
    plot_histogram(hw_counts[name],  ax=axes[i, 1], title=f"|{name}⟩ — {backend.name}")

plt.tight_layout()
plt.show()

---
## Part 2: Correlations in the X Basis (15 pts)

For each Bell state you already have $\langle ZZ\rangle$ from Part 1. Now measure $\langle XX\rangle$ by rotating both qubits with Hadamards immediately before measurement (that maps the X basis to the Z basis the hardware can actually measure).

**Correlation from counts (same formula in either basis):**
$$\langle O_1 O_2\rangle \;=\; P(00) + P(11) - P(01) - P(10)$$

**Theoretical values:**

| State | $\langle ZZ\rangle$ | $\langle XX\rangle$ |
|:-----:|:----:|:----:|
| $\lvert\Phi^+\rangle$ | $+1$ | $+1$ |
| $\lvert\Psi^-\rangle$ | $-1$ | $-1$ |

On a noiseless simulator you should see these values almost exactly. On real hardware, decoherence and gate errors will pull the magnitudes toward zero.

In [ ]:
def compute_correlation(counts):
    """
    Compute a two-qubit correlation ⟨O_1 O_2⟩ from measurement counts.
    Works for either basis as long as you measured in that basis.
    """
    # >>> INSTRUCTOR SOLUTION — remove before distributing >>>
    total = sum(counts.values())
    p = {k: v / total for k, v in counts.items()}
    return p.get('00', 0) + p.get('11', 0) - p.get('01', 0) - p.get('10', 0)
    # <<< END INSTRUCTOR SOLUTION <<<

# Sanity check on a perfect Φ+ simulator result
test = compute_correlation(sim_counts['Phi+'])
print(f"⟨ZZ⟩ for Φ+ (sim, Z basis): {test}")
assert test is not None and abs(test - 1.0) < 0.05, "⟨ZZ⟩ for Φ+ should be close to +1"
print("✓ compute_correlation looks right")

In [ ]:
# === X-basis measurement circuits ===
# Same Bell state preparation, but apply H to both qubits just before measuring.

def build_phi_plus_x():
    qc = QuantumCircuit(2, 2)
    qc.h(0); qc.cx(0, 1)
    qc.h(0); qc.h(1)          # rotate X basis → Z basis for readout
    qc.measure([0, 1], [0, 1])
    return qc

def build_psi_minus_x():
    """|Ψ-⟩ prep followed by H⊗H before measurement."""
    qc = QuantumCircuit(2, 2)
    # >>> INSTRUCTOR SOLUTION — remove before distributing >>>
    qc.h(0); qc.cx(0, 1); qc.x(1); qc.z(0)   # Ψ- prep
    qc.h(0); qc.h(1)                          # X-basis readout
    # <<< END INSTRUCTOR SOLUTION <<<
    qc.measure([0, 1], [0, 1])
    return qc

phi_plus_x  = build_phi_plus_x()
psi_minus_x = build_psi_minus_x()

In [ ]:
# === Run X-basis circuits on simulator ===
sim_counts_x = {
    'Phi+' : simulator.run(phi_plus_x,  shots=SHOTS).result().get_counts(),
    'Psi-' : simulator.run(psi_minus_x, shots=SHOTS).result().get_counts(),
}

xx_sim = {name: compute_correlation(counts) for name, counts in sim_counts_x.items()}
print(f"⟨XX⟩ for Φ+ (sim): {xx_sim['Phi+']:+.3f}   (theory = +1)")
print(f"⟨XX⟩ for Ψ- (sim): {xx_sim['Psi-']:+.3f}   (theory = -1)")

In [ ]:
# === Transpile and submit X-basis circuits to hardware ===
# >>> INSTRUCTOR SIM SUBSTITUTE — revert before distributing >>>
phi_plus_x_t  = pm.run(phi_plus_x)
psi_minus_x_t = pm.run(psi_minus_x)

hw_counts_x = {
    'Phi+': noisy_sim.run(phi_plus_x_t,  shots=SHOTS).result().get_counts(),
    'Psi-': noisy_sim.run(psi_minus_x_t, shots=SHOTS).result().get_counts(),
}
print(f"(noisy-sim substitute) Φ+ X-basis: {hw_counts_x['Phi+']}")
print(f"(noisy-sim substitute) Ψ- X-basis: {hw_counts_x['Psi-']}")
# <<< END INSTRUCTOR SIM SUBSTITUTE <<<

In [ ]:
# === Retrieve X-basis hardware results ===
# >>> INSTRUCTOR SIM SUBSTITUTE — see note on cell 12 >>>
for name, counts in hw_counts_x.items():
    print(f"{name} (X basis): {counts}")
# <<< END INSTRUCTOR SIM SUBSTITUTE <<<

In [ ]:
# === Comparison table ===
# Compute ZZ and XX on both simulator and hardware, compare to theory

theory = {
    'Phi+' : {'ZZ': +1, 'XX': +1},
    'Psi-' : {'ZZ': -1, 'XX': -1},
}

zz_sim = {name: compute_correlation(sim_counts[name])   for name in ['Phi+', 'Psi-']}
xx_sim = {name: compute_correlation(sim_counts_x[name]) for name in ['Phi+', 'Psi-']}
zz_hw  = {name: compute_correlation(hw_counts[name])    for name in ['Phi+', 'Psi-']}
xx_hw  = {name: compute_correlation(hw_counts_x[name])  for name in ['Phi+', 'Psi-']}

print(f"{'State':>6s}  {'⟨ZZ⟩ th':>8s}  {'⟨ZZ⟩ sim':>9s}  {'⟨ZZ⟩ hw':>9s}    "
      f"{'⟨XX⟩ th':>8s}  {'⟨XX⟩ sim':>9s}  {'⟨XX⟩ hw':>9s}")
print("-" * 78)
for name in ['Phi+', 'Psi-']:
    print(f"{name:>6s}  {theory[name]['ZZ']:>+8d}  {zz_sim[name]:>+9.3f}  {zz_hw[name]:>+9.3f}    "
          f"{theory[name]['XX']:>+8d}  {xx_sim[name]:>+9.3f}  {xx_hw[name]:>+9.3f}")

---
## Part 3: Analysis (15 pts)

Look up the backend properties (the cell below pulls the most relevant numbers), then answer in **4–6 sentences** in the writeup cell below.

1. How close are your measured correlations to the ideal values? Quote your numbers.
2. What is the dominant source of error on this backend? Reference specific gate error rates, $T_1$, $T_2$, and/or readout error from the cell below.
3. Did $\lvert\Phi^+\rangle$ and $\lvert\Psi^-\rangle$ have the same fidelity, or did one perform better? The two circuits differ by two extra single-qubit gates (X and Z) — is the degradation consistent with that?

In [ ]:
# === Backend properties for your analysis ===
# This pulls the device parameters for the qubits you used.

print(f"Backend: {backend.name}")
print(f"Number of qubits: {backend.num_qubits}")

# Per-qubit T1, T2 (via backend.target, the modern accessor)
target = backend.target
print("\nQubit properties (first 10 qubits):")
for q in range(min(10, backend.num_qubits)):
    qp = target.qubit_properties[q]
    t1_us = qp.t1 * 1e6 if qp.t1 else float('nan')
    t2_us = qp.t2 * 1e6 if qp.t2 else float('nan')
    print(f"  qubit {q}:  T1 = {t1_us:6.1f} us   T2 = {t2_us:6.1f} us")

# Readout error
print("\nReadout errors (first 10 qubits):")
for q in range(min(10, backend.num_qubits)):
    meas_props = target['measure'][(q,)]
    print(f"  qubit {q}: readout error = {meas_props.error:.1e}")

# Two-qubit gate errors (Heron r2 uses CZ as its native 2Q gate)
print("\nTwo-qubit gate errors (first 10 pairs):")
for gate_name in ['cz', 'ecr', 'cx']:
    if gate_name in target.operation_names:
        qargs_list = list(target.qargs_for_operation_name(gate_name))[:10]
        for qargs in qargs_list:
            err = target[gate_name][qargs].error
            dur = target[gate_name][qargs].duration * 1e9
            print(f"  {gate_name} on qubits {qargs}: error = {err:.2e}, duration = {dur:.0f} ns")
        break

**Your analysis:**

*(double-click to edit)*

**Backend used:**

**Backend specs (from the cell above):**
- Two-qubit gate error:
- $T_1$:
- $T_2$:
- Readout error:

**Analysis:**



---
## Submission Checklist

Before submitting, verify:

- [ ] Ψ⁻ circuit built correctly (statevector sanity check passes)
- [ ] Both Bell states run on the simulator (Z basis and X basis)
- [ ] Both Bell states run on real IBM hardware (Z basis and X basis)
- [ ] `compute_correlation` implemented and sanity-checked
- [ ] Comparison table printed showing theory vs. sim vs. hardware for both ⟨ZZ⟩ and ⟨XX⟩
- [ ] Histogram plots (simulator vs. hardware) included
- [ ] **IBM job IDs recorded below** (proof of hardware execution)
- [ ] Written analysis (4–6 sentences) with specific backend numbers

**IBM Job IDs:**

In [ ]:
# Record your IBM job IDs here (copy from the print statements above)
job_ids = {
    'part1_zz' : "",   # Z-basis job
    'part2_xx' : "",   # X-basis job
}
for label, jid in job_ids.items():
    print(f"  {label}: {jid}")

---
**End of Phase 1** — Good luck with the team project!